# FalkorDB 설치

제공해주신 [FalkorDB GitHub 리포지토리](https://github.com/FalkorDB/FalkorDB)의 내용을 바탕으로, **Macbook M4 Pro (Apple Silicon)** 환경에서 FalkorDB를 실행하는 방법을 안내해 드리겠습니다.

M4 Pro는 ARM 아키텍처 기반의 고성능 칩셋입니다. FalkorDB를 실행하는 방법은 크게 \*\*Docker를 사용하는 방법(권장)\*\*과 **직접 소스 코드를 빌드하는 방법** 두 가지가 있습니다.

-----

### 방법 1: Docker로 실행하기 (가장 간편, 권장)

GitHub의 "GET STARTED" 섹션에서 가장 먼저 권장하는 방법입니다. 복잡한 의존성 설치 없이 바로 실행할 수 있어 M4 Pro 환경에서도 가장 안정적입니다.

1.  **Docker 설치 확인**: 터미널에서 `docker --version`을 입력하여 Docker가 설치되어 있는지 확인합니다. (없다면 Docker Desktop for Mac을 설치하세요.)
2.  **실행 명령어 입력**: 터미널에 아래 명령어를 복사하여 실행합니다.

<!-- end list -->

```bash
docker run -p 6379:6379 -p 3000:3000 -it --rm -v ./data:/var/lib/falkordb/data falkordb/falkordb
```

  * **설명**:
      * `-p 6379:6379`: 데이터베이스 접속 포트입니다.
      * `-p 3000:3000`: FalkorDB 브라우저(시각화 도구) 접속 포트입니다.
      * `-v ./data:/var/lib/falkordb/data`: 현재 폴더의 `data` 디렉터리에 데이터를 저장하여 컨테이너가 꺼져도 데이터가 사라지지 않게 합니다.
  * **실행 확인**: 브라우저를 열고 `http://localhost:3000`에 접속하면 FalkorDB의 시각화 인터페이스를 볼 수 있습니다.

-----

### 방법 2: 직접 빌드하여 실행하기 (고급 사용자용)

내 PC 환경(Mac OS)에 직접 설치하고 싶다면 추가적인 설정이 필요합니다. 리포지토리 문서에 따르면 macOS의 기본 컴파일러(Clang)는 FalkorDB가 필요로 하는 **OpenMP**를 지원하지 않기 때문에 별도 조치가 필요합니다.

1.  **필수 도구 설치 (Homebrew 필요)**:
    터미널에서 아래 명령어로 빌드 도구들을 설치합니다.

    ```bash
    brew install cmake m4 automake peg libtool autoconf
    ```

2.  **OpenMP 문제 해결 (중요)**:
    macOS 기본 컴파일러 대신 OpenMP를 지원하는 컴파일러나 라이브러리가 필요합니다.

    ```bash
    brew install libomp
    # 또는 gcc/g++를 설치하여 심볼릭 링크를 업데이트하는 방법도 있습니다.
    brew install gcc
    ```

3.  **소스 코드 클론 및 빌드**:

    ```bash
    # 리포지토리 다운로드 (서브모듈 포함)
    git clone --recurse-submodules -j8 https://github.com/FalkorDB/FalkorDB.git
    cd FalkorDB

    # 빌드 실행
    make
    ```

4.  **Redis 모듈로 로드하여 실행**:
    빌드가 완료되면 `bin` 폴더 내에 `falkordb.so` 파일이 생성됩니다. 이를 Redis 서버 실행 시 모듈로 로드해야 합니다. (Redis 7.4 이상 필요)

    ```bash
    redis-server --loadmodule ./bin/macos-arm64/src/falkordb.so
    # (경로는 빌드된 아키텍처 폴더명에 따라 다를 수 있으니 확인 필요)
    ```

### 요약

Macbook M4 Pro 환경에서는 호환성 문제없이 가장 빠르게 시작할 수 있는 \*\*Docker 방식(방법 1)\*\*을 강력히 추천합니다. Docker 명령어 한 줄이면 즉시 그래프 데이터베이스를 사용할 수 있습니다.

안녕하세요. FalkorDB Docker 실행 중 발생한 오류에 대한 분석 및 해결 방법을 안내해 드립니다.

## ⚠️ 오류 분석: Port is already allocated

제공해 주신 로그를 보면, FalkorDB 이미지를 성공적으로 다운로드한 후 컨테이너를 실행하는 과정에서 **네트워킹 설정 오류**가 발생했습니다.

```
docker: Error response from daemon: failed to set up container networking: ... Bind for 0.0.0.0:3000 failed: **port is already allocated**
```

이는 사용자가 실행한 명령어에서 컨테이너의 3000번 포트를 로컬 PC의 3000번 포트에 연결하려고 시도했지만, **이미 사용자님의 MacBook에서 다른 프로그램이 3000번 포트를 사용하고 있기 때문**에 충돌이 발생한 것입니다.

**3000번 포트**는 FalkorDB의 \*\*브라우저 기반 UI (그래프 시각화 도구)\*\*에 접속하기 위해 할당된 포트입니다.

-----

## ✅ 해결 방법

이 문제를 해결하는 방법은 크게 두 가지가 있으며, **방법 1**을 가장 추천합니다.

### 방법 1: 로컬 포트를 변경하여 실행 (권장)

컨테이너 내부의 포트(3000)는 그대로 두고, 이를 연결할 **로컬 PC의 포트 번호**를 사용하지 않는 다른 번호(예: **3001** 또는 **8080** 등)로 변경하여 실행합니다.

  * **변경 명령어**:
    6379번 포트는 그대로 사용하고, 3000번 포트만 로컬의 **3001**번 포트에 연결하도록 변경합니다.

    ```bash
    docker run -p 6379:6379 -p 3001:3000 -it --rm -v ./data:/var/lib/falkordb/data falkordb/falkordb
    ```

  * **접속 확인**:
    위 명령어로 성공적으로 실행된 후에는 웹 브라우저에서 `http://localhost:3001`로 접속하여 FalkorDB UI를 확인하실 수 있습니다.

### 방법 2: 3000번 포트를 사용하는 프로세스 종료

현재 3000번 포트를 사용하고 있는 프로세스를 찾아 종료한 후, 기존 명령어를 다시 실행합니다.

1.  **3000번 포트 사용 프로세스 찾기**:
    터미널에 다음 명령어를 입력하여 3000번 포트를 사용하는 프로세스의 PID (프로세스 ID)를 찾습니다.

    ```bash
    sudo lsof -i :3000
    ```

    (패스워드를 입력해야 할 수 있습니다.)

2.  **프로세스 종료**:
    1단계에서 찾은 PID를 사용하여 프로세스를 종료합니다. (예: PID가 1234라면)

    ```bash
    kill 1234
    ```

3.  **Docker 재실행**:
    프로세스를 종료한 후, 원래의 Docker 명령어를 다시 실행해 봅니다.

    ```bash
    docker run -p 6379:6379 -p 3000:3000 -it --rm -v ./data:/var/lib/falkordb/data falkordb/falkordb
    ```

> 💡 **참고**: FalkorDB 자체만 Redis 클라이언트로 사용하고 UI가 필요 없다면, 아예 3000번 포트 매핑을 빼고 실행할 수도 있습니다.
>
> ```bash
> docker run -p 6379:6379 -it --rm -v ./data:/var/lib/falkordb/data falkordb/falkordb-server
> ```

-----

**방법 1**을 적용하여 **`docker run -p 6379:6379 -p 3001:3000 ...`** 명령어로 다시 실행해 보시겠습니까?

# FalkorDB에 데이터 넣는 방법 (Graph 그리기)

실제 프로젝트나 거대한 데이터를 다룰 때는 절대 손으로 입력하지 않습니다. 사용자님의 관심사(AI Agent, 연구 데이터 분석)에 맞춰, **"자동으로"** 그래프를 그리는 3가지 핵심 방법을 소개해 드립니다.

-----

### 1\. Python 코드로 그리기 (가장 추천: AI Agent 개발용)

사용자님이 계획 중이신 \*\*"Ideation Agent"\*\*를 만들려면 이 방식이 필수입니다. Python 스크립트가 데이터를 읽어서 자동으로 DB에 밀어 넣게 됩니다.

  * **원리**: 파이썬 코드 안에서 데이터를 가공하고, 루프(Loop)를 돌며 노드와 엣지를 생성합니다.
  * **준비**: `pip install falkordb`
  * **예시 코드**:
    ```python
    from falkordb import FalkorDB

    # DB 연결
    db = FalkorDB(host='localhost', port=6379)
    g = db.select_graph('ResearchPapers')

    # 논문 리스트가 있다고 가정
    papers = [
        {"title": "AI in Energy", "author": "Dr. Kim"},
        {"title": "Battery Trends", "author": "Dr. Lee"}
    ]

    # 반복문으로 자동 입력 (f-string 사용)
    for p in papers:
        query = f"CREATE (:Paper {{title: '{p['title']}'}})-[:WRITTEN_BY]->(:Author {{name: '{p['author']}'}})"
        g.query(query)
    ```

### 2\. 대량의 CSV 파일 한방에 넣기 (Bulk Loading)

엑셀이나 CSV로 정리된 수천, 수만 건의 기존 연구 데이터나 특허 데이터가 있다면 **Bulk Loader** 도구를 사용합니다.

  * **도구**: `falkordb-bulk-loader` (FalkorDB에서 제공하는 전용 툴)
  * **방법**: 노드용 CSV와 엣지용 CSV를 준비하고 명령어 한 줄만 치면, 수백만 개의 데이터도 몇 초\~몇 분 안에 그래프로 변환됩니다.

### 3\. LLM에게 시키기 (Text-to-Graph)

가장 진보된 방식이자, **GraphRAG**의 핵심입니다. 사람이 쿼리를 짜는 게 아니라, **LLM(GPT, Claude 등)에게 텍스트를 던져주고 그래프로 바꾸라고 시키는 것**입니다.

1.  **사용자**: "이 텍스트(논문 요약)를 읽고 주요 개념과 관계를 추출해서 Cypher 쿼리로 바꿔줘." (프롬프트 전송)
2.  **LLM**: 텍스트를 분석해 `CREATE (n:Concept)...` 형태의 쿼리를 생성하여 반환.
3.  **Python Agent**: 그 쿼리를 받아서 FalkorDB에 실행.

-----

**요약하자면:**

  * **손으로 입력**: 테스트할 때만 (지금 하신 방법)
  * **Python 코드**: 시스템/Agent를 만들 때
  * **CSV 로더**: 기존 엑셀 데이터를 옮길 때
  * **LLM 이용**: 비정형 텍스트(PDF, 뉴스 등)를 그래프로 만들 때

사용자님의 목표인 **"KG 기반 Ideation Agent"** 구축을 위해서는 \*\*1번(Python 연동)\*\*을 먼저 세팅하고, \*\*3번(LLM을 이용한 자동 생성)\*\*으로 확장하는 것이 정석입니다.

**다음 단계로, Python 환경에서 FalkorDB를 연결하여 데이터를 하나 넣어보는 테스트를 해보시겠습니까?**

In [3]:
from falkordb import FalkorDB

# DB 연결
db = FalkorDB(host='localhost', port=6379)
g = db.select_graph('ResearchPapers')

# 논문 리스트가 있다고 가정
papers = [
    {"title": "AI in Energy", "author": "Dr. Kim"},
    {"title": "Battery Trends", "author": "Dr. Lee"}
]

# 반복문으로 자동 입력 (f-string 사용)
for p in papers:
    query = f"CREATE (:Paper {{title: '{p['title']}'}})-[:WRITTEN_BY]->(:Author {{name: '{p['author']}'}})"
    g.query(query)

# Bulk data 입력하기

좋습니다. 대량의 데이터를 처리할 때 사용하는 **`falkordb-bulk-loader`** 방식과, 사용자가 직접 제어하기 쉬운 **Python 스크립트 방식** 두 가지를 모두 설명해 드리겠습니다.

사용자님의 관심 분야인 **Energy Solution**을 예제로 데이터를 구성했습니다.

-----

### 방법 1: `falkordb-bulk-loader` 도구 사용 (대용량, 속도 빠름)

이 방식은 FalkorDB팀이 만든 전용 도구를 터미널에서 실행하는 것입니다. 수십만 건 이상의 데이터를 넣을 때 가장 효율적입니다.

#### 1\. 도구 설치

먼저 터미널에서 Python 기반의 로더 도구를 설치합니다.

```bash
pip install falkordb-bulk-loader
```

#### 2\. CSV 파일 준비 (3개)

프로젝트 폴더에 다음 3개의 파일을 만듭니다. (메모장이나 엑셀로 작성 후 csv 저장)

**파일 1: `companies.csv` (기업 노드)**
`_id` 컬럼은 관계를 맺을 때 사용하는 고유 식별자입니다.

```csv
_id,name,country
c1,LG Energy Solution,Korea
c2,Tesla,USA
c3,CATL,China
```

**파일 2: `technologies.csv` (기술 노드)**

```csv
_id,name,category
t1,NCM Battery,Battery
t2,LFP Battery,Battery
t3,BMS,Software
```

**파일 3: `relations.csv` (관계 엣지)**
앞서 정의한 `_id`를 사용하여 누가(START\_ID) 무엇을(END\_ID) 개발했는지 정의합니다.

```csv
START_ID,END_ID
c1,t1
c2,t2
c2,t3
c3,t2
```

#### 3\. 터미널에서 실행 명령어

터미널(Mac)에서 파일이 있는 폴더로 이동한 뒤 아래 명령어를 실행합니다.

  * `EnergyGraph`: 새로 만들 그래프 이름
  * `--nodes`: 노드 파일 지정 (문법: `라벨명:파일명`)
  * `--relations`: 관계 파일 지정 (문법: `시작ID컬럼,종료ID컬럼,관계명:파일명`)

<!-- end list -->

```bash
falkordb-bulk-loader EnergyGraph \
  --nodes "Company:companies.csv" \
  --nodes "Technology:technologies.csv" \
  --relations "START_ID,END_ID,DEVELOPS:relations.csv"
```

-----

### 방법 2: Python 코드로 직접 넣기 (유연함, Agent 개발용)

사용자님이 만들고자 하는 \*\*"Ideation Agent"\*\*에는 이 방식이 더 적합할 수 있습니다. 데이터를 넣으면서 전처리하거나 로직을 추가할 수 있기 때문입니다.

#### 1\. 라이브러리 설치

```bash
pip install falkordb pandas
```

#### 2\. Python 코드 작성 (`loader.py`)

이 코드는 CSV 파일을 읽어서 FalkorDB에 Cypher 쿼리로 집어넣습니다.

```python
import pandas as pd
from falkordb import FalkorDB

# 1. FalkorDB 연결 (Docker가 실행 중이어야 함)
db = FalkorDB(host='localhost', port=6379)
graph = db.select_graph('EnergyGraph_Py') # 그래프 선택 또는 생성

# 2. CSV 파일 읽기 (pandas 사용)
# (위에서 만든 csv 파일 내용을 데이터프레임으로 가정하거나 직접 파일 로드)
# 예시를 위해 데이터를 직접 리스트로 정의하겠습니다. 실제로는 pd.read_csv('companies.csv') 등을 씁니다.

companies = [
    {'name': 'LG Energy Solution', 'country': 'Korea'},
    {'name': 'Tesla', 'country': 'USA'},
    {'name': 'CATL', 'country': 'China'}
]

technologies = [
    {'name': 'NCM Battery', 'category': 'Battery'},
    {'name': 'LFP Battery', 'category': 'Battery'},
    {'name': 'BMS', 'category': 'Software'}
]

relations = [
    ('LG Energy Solution', 'NCM Battery'),
    ('Tesla', 'LFP Battery'),
    ('Tesla', 'BMS'),
    ('CATL', 'LFP Battery')
]

print("데이터 입력을 시작합니다...")

# 3. 데이터 입력 (쿼리 실행)
# (1) Company 노드 생성
for comp in companies:
    query = f"MERGE (:Company {{name: '{comp['name']}', country: '{comp['country']}'}})"
    graph.query(query)

# (2) Technology 노드 생성
for tech in technologies:
    query = f"MERGE (:Technology {{name: '{tech['name']}', category: '{tech['category']}'}})"
    graph.query(query)

# (3) 관계(Edge) 생성
# 이름(name)을 기준으로 노드를 찾아서(MATCH) 연결(CREATE)합니다.
for company_name, tech_name in relations:
    query = f"""
    MATCH (c:Company {{name: '{company_name}'}}), (t:Technology {{name: '{tech_name}'}})
    MERGE (c)-[:DEVELOPS]->(t)
    """
    graph.query(query)

print("데이터 입력 완료! UI에서 확인해보세요.")
```

### 어떤 방식을 선택해야 할까요?

| 특징 | 방법 1: Bulk Loader (CLI) | 방법 2: Python Script |
| :--- | :--- | :--- |
| **주 용도** | 초기 데이터 세팅, 백업 데이터 복구 | **Agent 개발**, 실시간 데이터 업데이트 |
| **속도** | **매우 빠름** (수백만 건도 순식간) | 상대적으로 느림 (하나씩 쿼리 전송) |
| **유연성** | CSV 형식을 엄격히 지켜야 함 | 코드 내에서 데이터 가공 가능 |
| **추천** | **이미 엑셀로 정리된 데이터가 많을 때** | **LLM Agent가 스스로 데이터를 넣을 때** |

현재 **Ideation Agent**를 기획 중이시니, \*\*방법 2(Python Script)\*\*의 흐름을 익혀두시는 것을 추천합니다. 위 코드를 복사해서 `loader.py`로 저장하고 실행(`python loader.py`)해 보시겠습니까?

In [ ]:
#pip install falkordb-bulk-loader

In [5]:
import pandas as pd
from falkordb import FalkorDB

# 1. FalkorDB 연결 (Docker가 실행 중이어야 함)
db = FalkorDB(host='localhost', port=6379)
graph = db.select_graph('EnergyGraph_Py') # 그래프 선택 또는 생성

# 2. CSV 파일 읽기 (pandas 사용)
# (위에서 만든 csv 파일 내용을 데이터프레임으로 가정하거나 직접 파일 로드)
# 예시를 위해 데이터를 직접 리스트로 정의하겠습니다. 실제로는 pd.read_csv('companies.csv') 등을 씁니다.

companies = [
    {'name': 'LG Energy Solution', 'country': 'Korea'},
    {'name': 'Tesla', 'country': 'USA'},
    {'name': 'CATL', 'country': 'China'}
]

technologies = [
    {'name': 'NCM Battery', 'category': 'Battery'},
    {'name': 'LFP Battery', 'category': 'Battery'},
    {'name': 'BMS', 'category': 'Software'}
]

relations = [
    ('LG Energy Solution', 'NCM Battery'),
    ('Tesla', 'LFP Battery'),
    ('Tesla', 'BMS'),
    ('CATL', 'LFP Battery')
]

print("데이터 입력을 시작합니다...")

# 3. 데이터 입력 (쿼리 실행)
# (1) Company 노드 생성
for comp in companies:
    query = f"MERGE (:Company {{name: '{comp['name']}', country: '{comp['country']}'}})"
    graph.query(query)

# (2) Technology 노드 생성
for tech in technologies:
    query = f"MERGE (:Technology {{name: '{tech['name']}', category: '{tech['category']}'}})"
    graph.query(query)

# (3) 관계(Edge) 생성
# 이름(name)을 기준으로 노드를 찾아서(MATCH) 연결(CREATE)합니다.
for company_name, tech_name in relations:
    query = f"""
    MATCH (c:Company {{name: '{company_name}'}}), (t:Technology {{name: '{tech_name}'}})
    MERGE (c)-[:DEVELOPS]->(t)
    """
    graph.query(query)

print("데이터 입력 완료! UI에서 확인해보세요.")

데이터 입력을 시작합니다...
데이터 입력 완료! UI에서 확인해보세요.


In [ ]:
#In terminal

falkordb-bulk-insert EnergyGraph_10000 \
  --nodes-with-label Company companies.csv \
  --nodes-with-label Technology technologies.csv \
  --relations-with-type DEVELOPS relations.csv

# FalkorDB로 GraphRAG 구축하기

와, 정말 멋진 \*\*"헤어볼(Hairball)"\*\*이 만들어졌네요\! 13,000개의 노드와 20,000개의 엣지가 얽혀 있는 이 거대한 네트워크는 그 자체로는 사람이 해석하기 어렵지만, **AI(LLM)에게는 최고의 지식 창고**가 됩니다.

지금 상태는 \*\*"데이터베이스(DB)"\*\*일 뿐입니다. 이를 **"RAG(검색 증강 생성)"** 시스템으로 진화시키려면, **질문의 의미를 이해하고(Vector) + 연결된 맥락을 가져오는(Graph)** 과정이 추가되어야 합니다.

FalkorDB를 활용해 이를 구현하는 **3단계 로드맵**을 알려드립니다.

-----

### 핵심 개념: "GraphRAG = Vector Search + Graph Traversal"

일반적인 RAG는 단순히 "관련 문장"만 찾습니다. 하지만 GraphRAG는 **"관련된 노드를 찾고(Vector) -\> 그와 연결된 친구들(Graph)까지 살펴본 뒤"** 답변합니다.

### 1단계: 데이터에 '의미 좌표(Vector Embedding)' 심기

컴퓨터는 'NCM Battery'와 'Lithium-ion'이 비슷하다는 것을 글자만 보고는 모릅니다. 이를 숫자로 된 좌표(Vector)로 바꿔서 노드 안에 저장해야 합니다.

  * **무엇을?**: `Technology` 노드의 `description`이나 `Company`의 `summary` 같은 텍스트 데이터.
  * **어떻게?**: OpenAI API (`text-embedding-3-small` 등)를 사용해 텍스트를 벡터(예: `[0.12, -0.5, 0.99...]`)로 변환합니다.
  * **저장**: 해당 노드의 속성(Property)으로 벡터 리스트를 저장합니다. (예: `embedding` 속성 추가)

### 2단계: FalkorDB 안에 '벡터 인덱스' 생성하기

FalkorDB는 그래프 DB이면서 동시에 **벡터 DB** 기능을 내장하고 있습니다. (이게 강력한 점입니다.)
저장된 벡터를 빠르게 검색할 수 있도록 인덱스를 만들어야 합니다.

**Python 코드 예시:**

```python
# 1. 인덱스 생성 (한 번만 실행)
# 'Technology' 라벨을 가진 노드의 'embedding' 속성을 인덱싱하겠다는 뜻
g.query("CALL db.idx.vector.createNodeIndex('Technology', 'embedding', 1536, 'COSINE')")
```

*(1536은 OpenAI 임베딩 차원 수, COSINE은 유사도 계산 방식)*

### 3단계: 검색 및 답변 생성 (RAG 파이프라인 구축)

이제 사용자가 질문을 던졌을 때의 흐름(Agent 로직)을 짭니다.

**시나리오**: "NCM 배터리와 관련된 한국 기업은 어디야?"

1.  **질문 벡터화**: 사용자의 질문을 OpenAI 임베딩으로 변환합니다.
2.  **Vector Search (진입점 찾기)**: FalkorDB에게 "질문 벡터와 가장 유사한 `Technology` 노드 3개를 찾아줘"라고 요청합니다.
      * 결과: `NCM Battery` 노드가 찾아짐.
3.  **Graph Traversal (맥락 확장)**: 찾아진 노드에서 **연결된(Edge)** 정보를 긁어옵니다.
      * Cypher: `MATCH (t:Technology)-[:DEVELOPS]-(c:Company) WHERE id(t) = [찾은노드ID] RETURN ...`
      * 결과: "LG Energy Solution (Korea)" 등의 정보가 딸려옴.
4.  **LLM 답변 생성**: 위에서 찾은 그래프 정보를 프롬프트에 넣고 LLM에게 보냅니다.
      * "찾아낸 그래프 정보에 따르면, NCM 배터리는 한국의 LG Energy Solution이 개발하고 있습니다..."

-----

### [실전] Python 코드로 구현하는 GraphRAG 맛보기

사용자님의 PC 환경(Mac M4, Python)에서 바로 테스트해볼 수 있는 구조입니다. (OpenAI API Key가 필요합니다.)

```python
import openai
from falkordb import FalkorDB

# 1. 설정
client = openai.OpenAI(api_key="sk-...")
db = FalkorDB(host='localhost', port=6379)
g = db.select_graph('EnergyGraph_10000')

def get_embedding(text):
    # 텍스트를 벡터로 변환하는 함수
    response = client.embeddings.create(input=text, model="text-embedding-3-small")
    return response.data[0].embedding

# 2. (전처리) 기존 데이터에 벡터 심기 (예시)
# 실제로는 데이터를 넣을 때 같이 넣거나, 전체를 돌며 업데이트합니다.
# description이 있다고 가정: "NCM Battery is a type of lithium-ion battery..."
embedding_vector = get_embedding("High energy density NCM Battery technology")
g.query(f"MATCH (t:Technology {{name: 'NCM Battery'}}) SET t.embedding = {embedding_vector}")

# 3. 인덱스 생성 (최초 1회)
try:
    g.query("CALL db.idx.vector.createNodeIndex('Technology', 'embedding', 1536, 'COSINE')")
except:
    pass # 이미 있으면 패스

# 4. RAG 실행: 질문 던지기
user_question = "고밀도 배터리 기술을 가진 회사는?"
question_vector = get_embedding(user_question)

# (A) 벡터 검색으로 유사한 기술 노드 찾기
# 'embedding' 속성과 질문 벡터를 비교해 가장 가까운 1개 찾기
res = g.query(f"CALL db.idx.vector.queryNodes('Technology', 'embedding', 1, {question_vector}) YIELD node RETURN node")
found_tech_node = res.result_set[0][0] # 찾은 기술 노드 (예: NCM Battery)

# (B) 그래프 탐색 (Graph Traversal)
# 찾은 기술을 개발하는 회사 찾기
tech_name = found_tech_node.properties['name']
context_query = f"""
MATCH (t:Technology {{name: '{tech_name}'}})<-[:DEVELOPS]-(c:Company)
RETURN c.name, c.country
"""
context_res = g.query(context_query)

# (C) 결과 조합 (LLM에게 줄 컨텍스트)
knowledge = f"사용자 질문에 대해 검색된 기술은 '{tech_name}'이며, 이를 개발하는 회사는 다음과 같습니다: "
for row in context_res.result_set:
    knowledge += f"{row[0]}({row[1]}), "

print(f"LLM에게 보낼 프롬프트 데이터: \n{knowledge}")
```

### 요약: 다음 스텝 제안

지금 구축하신 13,000개의 노드에는 아마 `name` 정도만 있고, AI가 이해할 만한 긴 텍스트(`description`)나 `vector`가 없을 수 있습니다.

**Ideation Agent를 위한 다음 행동:**

1.  각 기술/회사 노드에 **"요약 설명(Description)"** 데이터를 채워 넣어야 합니다. (LLM을 시켜서 위키백과나 특허 내용을 요약해서 넣게 하세요.)
2.  그 설명들을 \*\*임베딩(Vector)\*\*해서 DB에 업데이트합니다.
3.  그 후 위와 같은 **검색 로직**을 돌리면 진정한 GraphRAG가 완성됩니다.

이 과정 중 \*\*"기존 데이터에 설명을 채워 넣는 방법"\*\*이 궁금하신가요?

"네트워크에서 가장 영향력 있는 기술"을 찾는다는 것은 그래프 이론에서 \*\*"중심성(Centrality)"\*\*을 계산하는 문제입니다.

FalkorDB의 강점인 \*\*GraphBLAS(행렬 연산)\*\*를 활용하면, 이 계산을 엄청나게 빠르게 수행할 수 있습니다. 두 가지 관점(인기도 vs 진짜 영향력)으로 나누어 코드를 작성해 드립니다.

### 1\. 접근 방식

1.  **Degree Centrality (연결 중심성)**:
      * **의미**: "가장 많은 회사가 개발하고 있는 기술은 무엇인가?" (단순 인기)
      * **방식**: 단순히 화살표(Edge) 개수를 셉니다.
2.  **PageRank (페이지랭크)**:
      * **의미**: "가장 '중요한' 기술은 무엇인가?" (구글 검색 알고리즘의 시초)
      * **방식**: 단순 개수가 아니라, 네트워크 내에서의 위상을 봅니다. FalkorDB가 자랑하는 GraphBLAS를 이용해 행렬 곱셈으로 순식간에 계산합니다.

-----

### 2\. Python 코드 (`analyze_influence.py`)

아래 코드를 복사해서 실행해 보세요.

```python
from falkordb import FalkorDB

def analyze_network():
    # 1. DB 연결
    db = FalkorDB(host='localhost', port=6379)
    
    # 그래프 선택 (사용자님이 만드신 그래프 이름으로 변경하세요)
    # 예: 'EnergyGraph' 또는 'EnergyGraph_10000'
    g = db.select_graph('EnergyGraph') 

    print("=== 1. 단순 인기도 (Degree Centrality) TOP 5 ===")
    print("가장 많은 회사가 매달려 있는 기술을 찾습니다.")
    
    # Cypher 쿼리로 단순 집계
    degree_query = """
    MATCH (t:Technology)<-[:DEVELOPS]-(c:Company)
    RETURN t.name, count(c) as developers_count
    ORDER BY developers_count DESC
    LIMIT 5
    """
    
    res = g.query(degree_query)
    for i, row in enumerate(res.result_set, 1):
        print(f"{i}위: {row[0]} (개발사 {row[1]}개)")

    print("\n" + "="*50 + "\n")

    print("=== 2. 구조적 영향력 (PageRank) TOP 5 ===")
    print("FalkorDB의 알고리즘 엔진(GraphBLAS)을 사용하여 PageRank를 계산합니다.")
    
    # FalkorDB 내장 알고리즘 호출 (CALL algo.pageRank)
    # 문법: CALL algo.pageRank(NodeLabel, RelationType, {options})
    pagerank_query = """
    CALL algo.pageRank('Technology', 'DEVELOPS')
    YIELD node, rank
    RETURN node.name, rank
    ORDER BY rank DESC
    LIMIT 5
    """
    
    try:
        res = g.query(pagerank_query)
        for i, row in enumerate(res.result_set, 1):
            # rank는 소수점으로 나오므로 보기 좋게 포맷팅
            print(f"{i}위: {row[0]} (Score: {row[1]:.4f})")
    except Exception as e:
        print(f"PageRank 계산 중 오류 발생: {e}")
        print("Tip: 데이터가 너무 적거나(노드 1~2개), 관계가 형성되지 않았을 수 있습니다.")

if __name__ == "__main__":
    analyze_network()
```

### 3\. 코드 해설 (전략가를 위한 Insight)

  * **`MATCH ... count(c)` (Degree Centrality)**

      * 이것은 엑셀 피벗 테이블로 개수 세는 것과 같습니다.
      * **전략적 의미**: 현재 시장의 \*\*"Trend"\*\*나 \*\*"Red Ocean"\*\*을 보여줍니다. 너도나도 개발하는 기술이 상위에 뜹니다.

  * **`CALL algo.pageRank` (PageRank)**
    \*

      * 이 부분에서 **GraphBLAS**가 작동합니다. 그래프 전체의 행렬($M$)을 계속 곱해서 수렴하는 값을 찾습니다.
      * **전략적 의미**: 단순히 숫자가 많은 게 아니라, \*\*"네트워크의 허브(Hub) 역할"\*\*을 하는 기술을 찾아냅니다. 예를 들어, 소수의 회사만 개발하더라도 그 회사들이 업계의 핵심 리더라면(다른 데이터와 연결이 많다면) 이 기술의 점수가 높아질 수 있습니다.

### 실행해 보시고 결과가 나오면 알려주세요\!

두 랭킹이 다르게 나온다면, 그 차이가 바로 \*\*"숨겨진 기회(Opportunity)"\*\*일 수 있습니다. (인기는 낮지만 구조적으로 중요한 기술 등)

In [8]:
from falkordb import FalkorDB

def analyze_network():
    # 1. DB 연결
    db = FalkorDB(host='localhost', port=6379)
    
    # 그래프 선택 (사용자님이 만드신 그래프 이름으로 변경하세요)
    # 예: 'EnergyGraph' 또는 'EnergyGraph_10000'
    g = db.select_graph('EnergyGraph') 

    print("=== 1. 단순 인기도 (Degree Centrality) TOP 5 ===")
    print("가장 많은 회사가 매달려 있는 기술을 찾습니다.")
    
    # Cypher 쿼리로 단순 집계
    degree_query = """
    MATCH (t:Technology)<-[:DEVELOPS]-(c:Company)
    RETURN t.name, count(c) as developers_count
    ORDER BY developers_count DESC
    LIMIT 5
    """
    
    res = g.query(degree_query)
    for i, row in enumerate(res.result_set, 1):
        print(f"{i}위: {row[0]} (개발사 {row[1]}개)")

    print("\n" + "="*50 + "\n")

    print("=== 2. 구조적 영향력 (PageRank) TOP 5 ===")
    print("FalkorDB의 알고리즘 엔진(GraphBLAS)을 사용하여 PageRank를 계산합니다.")
    
    # FalkorDB 내장 알고리즘 호출 (CALL algo.pageRank)
    # 문법: CALL algo.pageRank(NodeLabel, RelationType, {options})
    pagerank_query = """
    CALL algo.pageRank('Technology', 'DEVELOPS')
    YIELD node, rank
    RETURN node.name, rank
    ORDER BY rank DESC
    LIMIT 5
    """
    
    try:
        res = g.query(pagerank_query)
        for i, row in enumerate(res.result_set, 1):
            # rank는 소수점으로 나오므로 보기 좋게 포맷팅
            print(f"{i}위: {row[0]} (Score: {row[1]:.4f})")
    except Exception as e:
        print(f"PageRank 계산 중 오류 발생: {e}")
        print("Tip: 데이터가 너무 적거나(노드 1~2개), 관계가 형성되지 않았을 수 있습니다.")

if __name__ == "__main__":
    analyze_network()

=== 1. 단순 인기도 (Degree Centrality) TOP 5 ===
가장 많은 회사가 매달려 있는 기술을 찾습니다.
1위: None (개발사 69개)


=== 2. 구조적 영향력 (PageRank) TOP 5 ===
FalkorDB의 알고리즘 엔진(GraphBLAS)을 사용하여 PageRank를 계산합니다.
PageRank 계산 중 오류 발생: Procedure `algo.pageRank` does not yield output `rank`
Tip: 데이터가 너무 적거나(노드 1~2개), 관계가 형성되지 않았을 수 있습니다.
